In [ ]:
# CIR deep reference-ladder benchmark: standalone GPU/JAX Kaggle notebook.
# Role in the fleet: the reference-sensitivity LADDER for the strong-error
# benchmark — every scheme is measured against HH references at all rungs of
# CIR_SUITE_REFERENCE_GRID built from ONE shared fine Brownian path, so the
# fitted-order drift across rungs is pure reference effect. The production
# strong-error figures themselves come from kaggle_cir_benchmark_suite_JAX.
# The single cell contains JAX kernels and plotting/orchestration code directly.
# It does not clone, write, or import the local repository.
#
# 2026-07-10 revision (a), aligning with the post-2026-07-05 local pipeline:
#   1. ProjEuler floor: y_floor = dt was an absorbing trap for alpha < 0
#      (regime E strong error diverged; fitted order came out negative).
#      Now 0.5*sigma*sqrt(dt) = the HH truncation level sigma^2*dt/4 in X-space.
#   2. Canonical HH reference raised 16384 -> 32768 and coarse grid shifted to
#      2^-4..2^-9 (16..512 steps), matching configs/regimes.yaml.
#   3. Reference-sensitivity gate: every scheme error is measured against HH
#      references at ALL resolutions in CIR_SUITE_REFERENCE_GRID, built from
#      the SAME fine Brownian increments, so fitted-order drift across the
#      grid isolates reference contamination (Hefter--Jentzen: uniform-mesh
#      slopes above delta/2 are reference artefacts, not true rates).
#
# 2026-07-10 revision (b): TIME-STREAMED references for KLM-paper-depth grids
# (h_ref down to 2^-25) on a 16 GB P100. Full materialisation of the fine
# path is impossible there (2^25 steps = 268 MB/path), so the suite now uses
# three tiers, all driven by one fine Brownian path:
#   - Reference tier (up to 2^25): HH state advanced chunk-by-chunk
#     (CIR_SUITE_CHUNK_STEPS per chunk); only the per-path state is carried.
#   - Adaptive tier (CIR_SUITE_ADAPTIVE_GRID_STEPS, default 2^17): the
#     materialised grid consumed by KLM and adaptive KL. Their increments are
#     exact aggregates of the reference path; 2^17 is far below the smallest
#     admissible adaptive step h_min = 2^-9/64 = 2^-15, so the step-size
#     quantisation is unchanged in character.
#   - Coarse tier (16..512): fixed-step schemes, aggregated from the
#     adaptive tier.
#   Path batching (CIR_SUITE_MEM_BUDGET_GB, default 8) bounds device memory:
#   bytes/path ~ 8*(2*chunk + 2*adaptive). A streaming-vs-materialised HH
#   parity check runs at startup in every mode.
# RNG note: increments are drawn per (path-batch, time-chunk) via nested
# fold_in, so values differ from earlier revisions even at equal settings.
#
# Deep-ladder example (Kaggle P100):
#   CIR_SUITE_REFERENCE_GRID="32768,1048576,33554432"   # 2^15, 2^20, 2^25
#   CIR_SUITE_ADAPTIVE_GRID_STEPS=131072                # 2^17
#   CIR_SUITE_CHUNK_STEPS=32768

from __future__ import annotations

import csv
import os
import time
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.85")
os.environ.setdefault("JAX_ENABLE_X64", "true")

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

RUN_MODE = os.environ.get("CIR_SUITE_RUN_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError("CIR_SUITE_RUN_MODE must be 'full' or 'smoke'")

WORKING_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORKING_ROOT / "cir_reference_ladder_jax_outputs"
FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 120339106
FINAL_TIME = 1.0
SHARED = dict(kappa=2.0, theta=0.02, x0=0.02)
REGIMES = {
    "A": {"sigma": 0.10},
    "B": {"sigma": 0.20},
    "C": {"sigma": 0.282842712474619},
    "D": {"sigma": 0.50},
    "E": {"sigma": 0.80},
}

if RUN_MODE == "smoke":
    N_PATHS = 256
    REFERENCE_GRID = [128, 256, 512]
    COARSE_N_STEPS = [8, 16, 32]
    REGIME_LIST = ["A", "B", "C", "D", "E"]
    FIG3_CONFIG = dict(n_paths=64, reference_power=12, n_paper_a_values=5, levels=[4, 5])
    _default_adaptive = 256  # below ref_max, to exercise the tiering
    _default_chunk = 128     # 4 chunks, to exercise the streaming
else:
    N_PATHS = 20000
    REFERENCE_GRID = [
        int(v)
        for v in os.environ.get("CIR_SUITE_REFERENCE_GRID", "4096,16384,32768").split(",")
    ]
    # Back-compat: CIR_SUITE_REFERENCE_N_STEPS adds one more (typically deeper)
    # reference to the grid; the canonical reference is always the finest.
    _extra_ref = os.environ.get("CIR_SUITE_REFERENCE_N_STEPS")
    if _extra_ref:
        REFERENCE_GRID.append(int(_extra_ref))
    COARSE_N_STEPS = [16, 32, 64, 128, 256, 512]
    REGIME_LIST = ["A", "B", "C", "D", "E"]
    FIG3_CONFIG = dict(n_paths=1000, reference_power=int(os.environ.get("CIR_SUITE_FIG3_REFERENCE_POWER", "16")), n_paper_a_values=40, levels=[4, 5, 6, 7, 8, 9])
    _default_adaptive = None
    _default_chunk = None

REFERENCE_GRID = sorted(set(REFERENCE_GRID))
REFERENCE_N_STEPS = REFERENCE_GRID[-1]  # canonical (finest) reference

if _default_adaptive is None:
    _default_adaptive = min(REFERENCE_N_STEPS, 131072)
if _default_chunk is None:
    _default_chunk = min(REFERENCE_N_STEPS, 32768)
ADAPTIVE_GRID_STEPS = min(
    int(os.environ.get("CIR_SUITE_ADAPTIVE_GRID_STEPS", str(_default_adaptive))),
    REFERENCE_N_STEPS,
)
CHUNK_STEPS = min(
    int(os.environ.get("CIR_SUITE_CHUNK_STEPS", str(_default_chunk))),
    REFERENCE_N_STEPS,
)

for _ref in REFERENCE_GRID:
    if REFERENCE_N_STEPS % _ref != 0:
        raise ValueError("every reference in CIR_SUITE_REFERENCE_GRID must divide the finest one")
    if CHUNK_STEPS % (REFERENCE_N_STEPS // _ref) != 0:
        raise ValueError("CIR_SUITE_CHUNK_STEPS must be a multiple of ref_max/ref for every ladder rung")
if REFERENCE_N_STEPS % ADAPTIVE_GRID_STEPS != 0:
    raise ValueError("the adaptive grid must divide the finest reference")
if CHUNK_STEPS % (REFERENCE_N_STEPS // ADAPTIVE_GRID_STEPS) != 0:
    raise ValueError("CIR_SUITE_CHUNK_STEPS must be a multiple of ref_max/adaptive_grid")
if REFERENCE_N_STEPS % CHUNK_STEPS != 0:
    raise ValueError("CIR_SUITE_CHUNK_STEPS must divide the finest reference")
for _n in COARSE_N_STEPS:
    if ADAPTIVE_GRID_STEPS % _n != 0:
        raise ValueError("the adaptive grid must be divisible by every coarse level")


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def fit_loglog_order(step_sizes, errors):
    h = np.asarray(step_sizes, dtype=float)
    e = np.maximum(np.asarray(errors, dtype=float), 1e-300)
    return float(np.polyfit(np.log(h), np.log(e), 1)[0])


def cir_delta(kappa, theta, sigma):
    return 4.0 * kappa * theta / sigma**2


def kl_alpha(kappa, theta, sigma):
    return (4.0 * kappa * theta - sigma**2) / 8.0


def aggregate_dW(dW_fine, factor):
    n_paths, n_fine = dW_fine.shape
    return dW_fine.reshape((n_paths, n_fine // factor, factor)).sum(axis=2)


def hh_step(x, kappa, theta, sigma, dt, dW):
    floor = 0.25 * sigma**2 * dt
    r1 = jnp.maximum(
        0.5 * sigma * jnp.sqrt(dt),
        jnp.sqrt(jnp.maximum(floor, x)) + 0.5 * sigma * dW,
    )
    x_hat = r1 * r1 + dt * (kappa * (theta - x) - 0.25 * sigma**2)
    return jnp.maximum(x_hat, 0.0)


def hh_advance_state(x, kappa, theta, sigma, dt, dW):
    """Advance HH from state vector x through the increments dW (streaming)."""
    def step(x_c, dW_col):
        return hh_step(x_c, kappa, theta, sigma, dt, dW_col), None
    x_out, _ = jax.lax.scan(step, x, dW.T)
    return x_out


_hh_advance = jax.jit(hh_advance_state)


def hh_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    return hh_advance_state(x0, kappa, theta, sigma, dt, dW)


def fte_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    def step(x_aux, dW_col):
        x_pos = jnp.maximum(x_aux, 0.0)
        x_next = x_aux + kappa * (theta - x_pos) * dt + sigma * jnp.sqrt(x_pos) * dW_col
        return x_next, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x_aux, _ = jax.lax.scan(step, x0, dW.T)
    return jnp.maximum(x_aux, 0.0)


def projected_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    # 2026-07-05 floor fix: y_floor = dt is an absorbing trap for alpha < 0
    # (escape needs a normal draw > 2|alpha|/(sigma*sqrt(dt))); the Lamperti-
    # scale floor 0.5*sigma*sqrt(dt) matches the HH truncation level.
    y_floor = 0.5 * sigma * np.sqrt(dt)
    def step(y, dW_col):
        y_safe = jnp.maximum(y, y_floor)
        y_hat = y_safe + (alpha / y_safe - 0.5 * kappa * y_safe) * dt + 0.5 * sigma * dW_col
        return jnp.maximum(y_hat, y_floor), None
    y0 = jnp.full((dW.shape[0],), max(np.sqrt(X0), y_floor), dtype=jnp.float64)
    y, _ = jax.lax.scan(step, y0, dW.T)
    return y * y


def kl_uniform_terminal_from_dW_jax(X0, kappa, theta, sigma, dt, dW):
    alpha = kl_alpha(kappa, theta, sigma)
    decay = np.exp(-kappa * dt)
    def step(x, dW_col):
        return decay * (jnp.sqrt(x + 2.0 * alpha * dt) + 0.5 * sigma * dW_col) ** 2, None
    x0 = jnp.full((dW.shape[0],), X0, dtype=jnp.float64)
    x, _ = jax.lax.scan(step, x0, dW.T)
    return x


def soft_zero_threshold(kappa, theta, dt_max, rho=2.0):
    return theta * (1.0 - np.exp(-kappa * dt_max)) / rho


def kl_adaptive_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, dt_max, dW_fine, rho=2.0, safety=0.95):
    alpha = kl_alpha(kappa, theta, sigma)
    X_zero = soft_zero_threshold(kappa, theta, dt_max, rho)
    dW_fine = jnp.asarray(dW_fine, dtype=jnp.float64)
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    x0 = jnp.full((n_paths,), X0, dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(2, dtype=jnp.int64)

    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)

    def body(state):
        x, pos, counters = state
        active = pos < n_fine
        remaining_steps = n_fine - pos
        remaining_time = remaining_steps * dt_fine
        in_soft_zero = active & (x < X_zero)
        in_splitting = active & (~in_soft_zero)

        x_for_soft = jnp.minimum(x, X_zero * 0.999999)
        dt_sz = -jnp.log((X_zero - theta) / (x_for_soft - theta)) / kappa
        if alpha < 0.0:
            dt_adaptive = safety * x / (2.0 * abs(alpha))
            split_h = jnp.minimum(jnp.minimum(dt_adaptive, dt_max), remaining_time)
        else:
            split_h = jnp.minimum(dt_max, remaining_time)

        h_cont = jnp.where(in_soft_zero, jnp.minimum(dt_sz, remaining_time), 0.0)
        h_cont = jnp.where(in_splitting, split_h, h_cont)
        m = jnp.floor(h_cont / dt_fine).astype(jnp.int64)
        m = jnp.where(active, jnp.maximum(m, 1), 0)
        m = jnp.minimum(m, remaining_steps)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]

        decay = jnp.exp(-kappa * h)
        x_soft = decay * x + theta * (1.0 - decay)
        inside_sqrt = jnp.maximum(x + 2.0 * alpha * h, 0.0)
        x_split = decay * (jnp.sqrt(inside_sqrt) + 0.5 * sigma * dW) ** 2
        x_next = jnp.where(in_soft_zero, x_soft, jnp.where(in_splitting, x_split, x))

        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(in_soft_zero)],
            dtype=jnp.int64,
        )
        return x_next, pos + m, counters

    x, _, counters = jax.lax.while_loop(cond, body, (x0, pos0, counters0))
    total = int(counters[0])
    return x, {
        "n_steps_total": total,
        "mean_steps_per_path": total / n_paths,
        "soft_zero_fraction": int(counters[1]) / max(total, 1),
    }


def implicit_step(y, h, dW, alpha, beta, gamma):
    a = 1.0 - beta * h
    u = y + gamma * dW
    return (u + jnp.sqrt(u * u + 4.0 * a * alpha * h)) / (2.0 * a)


def projected_backstop_step(y, h, dW, alpha, beta, gamma):
    y_floor = gamma * jnp.sqrt(h)
    y_hat = y + (alpha / y + beta * y) * h + gamma * dW
    return jnp.maximum(y_hat, y_floor)


def klm_terminal_from_fine_dW_jax(X0, kappa, theta, sigma, T, h_max, dW_fine, rho=64.0):
    alpha = kl_alpha(kappa, theta, sigma)
    beta = -0.5 * kappa
    gamma = 0.5 * sigma
    use_implicit = alpha > 0.0
    n_paths, n_fine = dW_fine.shape
    dt_fine = T / n_fine
    h_min = h_max / rho
    m_min = max(int(round(h_min / dt_fine)), 1)
    m_max = max(int(np.floor(h_max / dt_fine)), 1)
    W = jnp.concatenate([jnp.zeros((n_paths, 1), dtype=jnp.float64), jnp.cumsum(dW_fine, axis=1)], axis=1)
    rows = jnp.arange(n_paths)
    y0 = jnp.full((n_paths,), jnp.sqrt(X0), dtype=jnp.float64)
    pos0 = jnp.zeros((n_paths,), dtype=jnp.int64)
    counters0 = jnp.zeros(3, dtype=jnp.int64)
    def cond(state):
        _, pos, _ = state
        return jnp.any(pos < n_fine)
    def body(state):
        y, pos, counters = state
        active = pos < n_fine
        h_prop = h_max * jnp.minimum(1.0, jnp.abs(y))
        m = jnp.floor(h_prop / dt_fine).astype(jnp.int64)
        min_triggered = m < m_min
        m = jnp.where(min_triggered, m_min, jnp.minimum(m, m_max))
        m = jnp.minimum(m, n_fine - pos)
        h = m * dt_fine
        dW = W[rows, pos + m] - W[rows, pos]
        y_explicit = jnp.where(h > 0.0, y + (alpha / y + beta * y) * h + gamma * dW, y)
        y_imp = implicit_step(y, h, dW, alpha, beta, gamma)
        y_proj = projected_backstop_step(y, h, dW, alpha, beta, gamma)
        y_backstop = jnp.where(use_implicit, y_imp, y_proj)
        neg_retake = (~min_triggered) & (y_explicit <= 0.0)
        use_backstop = min_triggered | neg_retake
        y_next = jnp.where(use_backstop, y_backstop, y_explicit)
        counters = counters + jnp.array(
            [jnp.sum(active), jnp.sum(active & min_triggered), jnp.sum(active & neg_retake)],
            dtype=jnp.int64,
        )
        return y_next, pos + m, counters
    y, _, counters = jax.lax.while_loop(cond, body, (y0, pos0, counters0))
    total = int(counters[0])
    return y * y, {
        "n_steps_total": total,
        "n_backstop_min": int(counters[1]),
        "n_backstop_neg": int(counters[2]),
        "backstop_fraction": (int(counters[1]) + int(counters[2])) / max(total, 1),
    }


def resolve_path_batch_sizes(n_paths):
    """Split paths into near-equal batches within the device-memory budget.

    Peak co-resident buffers per path: the fine chunk and its scan transpose
    (2 * CHUNK_STEPS) plus the adaptive-tier buffer and the KLM/KL cumsum W
    (2 * ADAPTIVE_GRID_STEPS), all float64.
    """
    override = os.environ.get("CIR_SUITE_PATH_BATCH")
    if override:
        cap = max(1, min(n_paths, int(override)))
    else:
        budget_bytes = float(os.environ.get("CIR_SUITE_MEM_BUDGET_GB", "8")) * 1e9
        bytes_per_path = 8.0 * (2 * CHUNK_STEPS + 2 * ADAPTIVE_GRID_STEPS)
        cap = max(1, min(n_paths, int(budget_bytes // bytes_per_path)))
    n_batches = -(-n_paths // cap)
    base, rem = divmod(n_paths, n_batches)
    return [base + (1 if i < rem else 0) for i in range(n_batches)]


def batch_chunk_key(batch_index, chunk_index):
    key = jax.random.fold_in(jax.random.PRNGKey(MASTER_SEED), batch_index)
    return jax.random.fold_in(key, chunk_index)


def streaming_parity_check():
    """Streamed HH (chunked, carried state) must equal the single-pass scan."""
    kappa, theta, x0 = SHARED["kappa"], SHARED["theta"], SHARED["x0"]
    sigma = REGIMES["C"]["sigma"]
    n_paths, ref_n, chunk = 8, 64, 16
    dt = FINAL_TIME / ref_n
    chunks = []
    x = jnp.full((n_paths,), x0, dtype=jnp.float64)
    for chunk_index in range(ref_n // chunk):
        key = batch_chunk_key(0, chunk_index)
        dW = jax.random.normal(key, (n_paths, chunk), dtype=jnp.float64) * jnp.sqrt(dt)
        chunks.append(dW)
        x = hh_advance_state(x, kappa, theta, sigma, dt, dW)
    full = hh_terminal_from_dW_jax(x0, kappa, theta, sigma, dt, jnp.concatenate(chunks, axis=1))
    err = float(jnp.max(jnp.abs(x - full)))
    if not err < 1e-12:
        raise RuntimeError(f"streaming parity check FAILED: max |diff| = {err:.3e}")
    print(f"streaming parity check passed (max |diff| = {err:.2e})")


def scheme_terminal(scheme, sigma, n_steps, dW_adaptive, adaptive_steps):
    """Terminal values for one scheme at one coarse level, plus step stats."""
    dt = FINAL_TIME / n_steps
    batch_n = dW_adaptive.shape[0]
    if scheme == "FTE":
        dW = aggregate_dW(dW_adaptive, adaptive_steps // n_steps)
        terminal = fte_terminal_from_dW_jax(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, dt, dW)
        return terminal, dict(variant="fixed", steps=batch_n * n_steps, backstop=0)
    if scheme == "ProjEuler":
        dW = aggregate_dW(dW_adaptive, adaptive_steps // n_steps)
        terminal = projected_terminal_from_dW_jax(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, dt, dW)
        return terminal, dict(variant="fixed", steps=batch_n * n_steps, backstop=0)
    if scheme == "KL":
        if kl_alpha(SHARED["kappa"], SHARED["theta"], sigma) < 0.0:
            terminal, stats = kl_adaptive_terminal_from_fine_dW_jax(
                SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma,
                FINAL_TIME, dt, dW_adaptive,
            )
            soft = int(round(stats["soft_zero_fraction"] * stats["n_steps_total"]))
            return terminal, dict(variant="adaptive-soft-zero", steps=stats["n_steps_total"], backstop=soft)
        dW = aggregate_dW(dW_adaptive, adaptive_steps // n_steps)
        terminal = kl_uniform_terminal_from_dW_jax(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, dt, dW)
        return terminal, dict(variant="uniform", steps=batch_n * n_steps, backstop=0)
    terminal, stats = klm_terminal_from_fine_dW_jax(SHARED["x0"], SHARED["kappa"], SHARED["theta"], sigma, FINAL_TIME, dt, dW_adaptive)
    return terminal, dict(
        variant="adaptive-backstop",
        steps=stats["n_steps_total"],
        backstop=stats["n_backstop_min"] + stats["n_backstop_neg"],
    )


def run_strong_error_suite() -> list[dict]:
    ref_max = REFERENCE_N_STEPS
    adaptive_steps = ADAPTIVE_GRID_STEPS
    n_chunks = ref_max // CHUNK_STEPS
    dt_fine = FINAL_TIME / ref_max
    schemes = ["FTE", "ProjEuler", "KL", "KLM"]
    batch_sizes = resolve_path_batch_sizes(N_PATHS)
    print(
        f"strong-error suite: reference grid {REFERENCE_GRID} (canonical {ref_max}), "
        f"adaptive tier {adaptive_steps}, chunk {CHUNK_STEPS} x {n_chunks}, "
        f"path batches {batch_sizes}"
    )

    all_rows = []
    sensitivity_rows = []
    order_rows = []
    for regime_name in REGIME_LIST:
        sigma = REGIMES[regime_name]["sigma"]
        acc = {
            (scheme, n_steps): {
                "abs_sum": {ref: 0.0 for ref in REFERENCE_GRID},
                "sq_sum": {ref: 0.0 for ref in REFERENCE_GRID},
                "runtime": 0.0,
                "steps": 0,
                "backstop": 0,
                "variant": None,
            }
            for scheme in schemes
            for n_steps in COARSE_N_STEPS
        }
        for batch_index, batch_n in enumerate(batch_sizes):
            # Stream the fine Brownian path: advance every reference-ladder HH
            # state chunk-by-chunk and materialise only the adaptive tier.
            ref_state = {
                ref_n: jnp.full((batch_n,), SHARED["x0"], dtype=jnp.float64)
                for ref_n in REFERENCE_GRID
            }
            adaptive_slices = []
            adaptive_factor = ref_max // adaptive_steps
            for chunk_index in range(n_chunks):
                key = batch_chunk_key(batch_index, chunk_index)
                dW_chunk = jax.random.normal(key, (batch_n, CHUNK_STEPS), dtype=jnp.float64) * jnp.sqrt(dt_fine)
                for ref_n in REFERENCE_GRID:
                    factor = ref_max // ref_n
                    dW_ref = dW_chunk if factor == 1 else aggregate_dW(dW_chunk, factor)
                    ref_state[ref_n] = _hh_advance(
                        ref_state[ref_n], SHARED["kappa"], SHARED["theta"], sigma,
                        FINAL_TIME / ref_n, dW_ref,
                    )
                adaptive_slices.append(
                    dW_chunk if adaptive_factor == 1 else aggregate_dW(dW_chunk, adaptive_factor)
                )
            dW_adaptive = (
                adaptive_slices[0] if n_chunks == 1 else jnp.concatenate(adaptive_slices, axis=1)
            )
            del adaptive_slices
            references = {}
            for ref_n, state in ref_state.items():
                state.block_until_ready()
                if not jnp.all(jnp.isfinite(state)):
                    raise RuntimeError(f"non-finite HH reference (regime {regime_name}, ref {ref_n})")
                references[ref_n] = state
            for scheme in schemes:
                for n_steps in COARSE_N_STEPS:
                    entry = acc[(scheme, n_steps)]
                    start = time.perf_counter()
                    terminal, stats = scheme_terminal(scheme, sigma, n_steps, dW_adaptive, adaptive_steps)
                    terminal.block_until_ready()
                    entry["runtime"] += time.perf_counter() - start
                    entry["steps"] += stats["steps"]
                    entry["backstop"] += stats["backstop"]
                    entry["variant"] = stats["variant"]
                    for ref_n, ref_terminal in references.items():
                        diff = terminal - ref_terminal
                        entry["abs_sum"][ref_n] += float(jnp.sum(jnp.abs(diff)))
                        entry["sq_sum"][ref_n] += float(jnp.sum(diff * diff))
        regime_rows = []
        for scheme in schemes:
            for n_steps in COARSE_N_STEPS:
                entry = acc[(scheme, n_steps)]
                for ref_n in REFERENCE_GRID:
                    row = {
                        "regime": regime_name,
                        "scheme": scheme,
                        "scheme_variant": entry["variant"],
                        "reference": "HH",
                        "reference_n_steps": ref_n,
                        "adaptive_grid_steps": adaptive_steps,
                        "dt": FINAL_TIME / n_steps,
                        "l1": entry["abs_sum"][ref_n] / N_PATHS,
                        "l2": float(np.sqrt(entry["sq_sum"][ref_n] / N_PATHS)),
                        "runtime_s": entry["runtime"],
                        "mean_steps_per_path": entry["steps"] / N_PATHS,
                        "backstop_fraction": entry["backstop"] / max(entry["steps"], 1),
                    }
                    sensitivity_rows.append(row)
                    if ref_n == ref_max:
                        regime_rows.append(row)
        all_rows.extend(regime_rows)
        write_csv(RES_DIR / f"jax_strong_error_regime_{regime_name}.csv", regime_rows)
        for scheme in schemes:
            for ref_n in REFERENCE_GRID:
                sr = sorted(
                    (
                        r for r in sensitivity_rows
                        if r["regime"] == regime_name
                        and r["scheme"] == scheme
                        and r["reference_n_steps"] == ref_n
                    ),
                    key=lambda r: r["dt"],
                )
                dts = [r["dt"] for r in sr]
                l2s = [r["l2"] for r in sr]
                order_rows.append({
                    "sensitivity_kind": "strong_error",
                    "regime": regime_name,
                    "scheme": scheme,
                    "reference_n_steps": ref_n,
                    "fitted_l1_order": fit_loglog_order(dts, [r["l1"] for r in sr]),
                    "fitted_l2_order": fit_loglog_order(dts, l2s),
                    # tail fit drops the coarsest (largest-h) level
                    "fitted_l2_order_tail": fit_loglog_order(dts[:-1], l2s[:-1]),
                })
        fig, ax = plt.subplots(figsize=(6.0, 4.4))
        for scheme in schemes:
            sr = [r for r in regime_rows if r["scheme"] == scheme]
            if not sr:
                continue
            order = fit_loglog_order([r["dt"] for r in sr], [r["l2"] for r in sr])
            ax.loglog([r["dt"] for r in sr], [r["l2"] for r in sr], "o-", label=f"{scheme} ({order:.2f})")
        ax.set_xlabel("step size h")
        ax.set_ylabel("strong L2 error")
        ax.set_title(f"JAX strong benchmark, regime {regime_name} (HH ref {ref_max})")
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"jax_strong_error_regime_{regime_name}.pdf")
        plt.close(fig)
    write_csv(RES_DIR / "jax_strong_error_all_references.csv", sensitivity_rows)
    write_csv(RES_DIR / "strong_reference_sensitivity_orders.csv", order_rows)
    make_order_summary_figure(order_rows)
    for row in order_rows:
        if row["reference_n_steps"] == REFERENCE_N_STEPS:
            print(
                f"{row['regime']}  {row['scheme']:<9} "
                f"L2 order full={row['fitted_l2_order']:.3f} "
                f"tail={row['fitted_l2_order_tail']:.3f}"
            )
    return all_rows


def make_order_summary_figure(order_rows) -> None:
    """Order-vs-delta summary with the HJ ceiling and reference-sensitivity bars.

    Points are tail-fit L2 orders against the canonical (finest) reference;
    vertical bars span the full-fit order across the whole reference grid.
    """
    ref_max = REFERENCE_N_STEPS
    styles = {
        "FTE": dict(color="#4477AA", marker="o", label="Full truncation Euler"),
        "ProjEuler": dict(color="#AA3377", marker="^", label="Projected Euler"),
        "KL": dict(color="#EE6677", marker="d", label="Kelly–Lord splitting"),
        "KLM": dict(color="#7B1FA2", marker="v", label="KLM backstopped adaptive"),
    }
    deltas = {
        name: cir_delta(SHARED["kappa"], SHARED["theta"], REGIMES[name]["sigma"])
        for name in REGIME_LIST
    }
    regime_order = sorted(REGIME_LIST, key=lambda n: deltas[n])

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    band = np.linspace(min(deltas.values()) * 0.8, 2.0, 100)
    ax.fill_between(band, band / 2.0, 1.55, color="0.88", zorder=0)
    ax.plot(band, band / 2.0, color="k", lw=1.4, zorder=1)
    ax.annotate(
        "Hefter–Jentzen ceiling $\\delta/2$\n(uniform-mesh schemes)",
        xy=(0.55, 0.68), fontsize=8, ha="center",
    )
    ax.axvline(2.0, color="k", ls=":", lw=0.9)
    ax.annotate(
        "Feller boundary\n$\\delta = 2$", xy=(2.0, 1.47), fontsize=8,
        ha="center", va="top",
    )
    ax.axhline(0.5, color="0.5", ls="--", lw=0.8, zorder=1)
    ax.axhline(1.0, color="0.5", ls=":", lw=0.8, zorder=1)
    ax.axhline(0.0, color="0.5", lw=0.8, zorder=1)

    for scheme, style in styles.items():
        xs, ys = [], []
        for name in regime_order:
            pts = [r for r in order_rows if r["regime"] == name and r["scheme"] == scheme]
            if not pts:
                continue
            canonical = next(r for r in pts if r["reference_n_steps"] == ref_max)
            xs.append(deltas[name])
            ys.append(canonical["fitted_l2_order_tail"])
            lo = min(r["fitted_l2_order"] for r in pts)
            hi = max(r["fitted_l2_order"] for r in pts)
            ax.vlines(deltas[name], lo, hi, color=style["color"], lw=3.0, alpha=0.30, zorder=1)
        if xs:
            ax.plot(xs, ys, lw=1.2, ms=6, **style)

    ax.set_xscale("log")
    ax.set_xticks([deltas[n] for n in regime_order])
    ax.set_xticklabels(
        [f"{deltas[n]:g}\n({n})" for n in regime_order], fontsize=8,
    )
    ax.set_xlabel(r"Bessel dimension $\delta = 4\kappa\theta/\sigma^2$  (regime)")
    ax.set_ylabel(r"fitted strong $L^2$ order (tail fit)")
    ax.set_ylim(-0.35, 1.55)
    grid_txt = ", ".join(str(r) for r in REFERENCE_GRID)
    fig.text(
        0.01,
        0.005,
        f"Shaded bars: fitted-order span across HH references {grid_txt} on the "
        "same Brownian path. Uniform-mesh points above the $\\delta/2$ band are "
        "reference-limited coupled diagnostics, not true rates.",
        fontsize=6.5,
        color="0.35",
    )
    ax.set_title("Observed strong convergence order across the regime grid")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(True, which="major", axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "jax_fig_order_vs_delta_summary.pdf")
    fig.savefig(FIG_DIR / "jax_fig_order_vs_delta_summary.png", dpi=150)
    plt.close(fig)


def sigma_from_a(a, kappa, lam):
    return np.sqrt(2.0 * kappa * lam * np.asarray(a, dtype=float))


def run_fig3_order_sweep() -> list[dict]:
    rows = []
    lambda_value = 0.05
    a_values = np.concatenate(([0.0], np.linspace(0.04, 1.6, FIG3_CONFIG["n_paper_a_values"])))
    for kappa in [2.0, 0.2]:
        reference_power = FIG3_CONFIG["reference_power"]
        n_fine = 2**reference_power
        dt_fine = FINAL_TIME / n_fine
        dW_fine = jax.random.normal(
            jax.random.PRNGKey(MASTER_SEED),
            (FIG3_CONFIG["n_paths"], n_fine),
            dtype=jnp.float64,
        ) * jnp.sqrt(dt_fine)
        for a in a_values:
            sigma = float(sigma_from_a(a, kappa, lambda_value))
            reference = hh_terminal_from_dW_jax(0.02**2, kappa, lambda_value, sigma, dt_fine, dW_fine)
            reference.block_until_ready()
            for level in FIG3_CONFIG["levels"]:
                hmax = 2.0 ** (-level)
                start = time.perf_counter()
                terminal, stats = klm_terminal_from_fine_dW_jax(0.02**2, kappa, lambda_value, sigma, FINAL_TIME, hmax, dW_fine)
                terminal.block_until_ready()
                runtime = time.perf_counter() - start
                diff = terminal - reference
                rows.append({
                    "kappa": kappa,
                    "a": float(a),
                    "sigma": sigma,
                    "level": level,
                    "hmax": hmax,
                    "reference_power": reference_power,
                    "n_paths": FIG3_CONFIG["n_paths"],
                    "rmse": float(jnp.sqrt(jnp.mean(diff * diff))),
                    "backstop_fraction": float(stats["backstop_fraction"]),
                    "mean_steps_per_path": float(stats["n_steps_total"] / FIG3_CONFIG["n_paths"]),
                    "runtime_s": runtime,
                })
    write_csv(RES_DIR / "jax_fig3_order_sweep.csv", rows)
    orders = []
    for kappa, a in sorted({(r["kappa"], r["a"]) for r in rows}):
        sr = [r for r in rows if r["kappa"] == kappa and r["a"] == a]
        if len(sr) >= 2:
            orders.append({"kappa": kappa, "a": a, "fitted_order": fit_loglog_order([r["hmax"] for r in sr], [r["rmse"] for r in sr])})
    write_csv(RES_DIR / "jax_fig3_fitted_orders.csv", orders)
    fig, ax = plt.subplots(figsize=(6.0, 4.4))
    for kappa in [2.0, 0.2]:
        sr = [r for r in orders if r["kappa"] == kappa and r["a"] > 0.0]
        ax.plot([r["a"] for r in sr], [r["fitted_order"] for r in sr], "o-", label=f"kappa={kappa:g}")
    ax.axhline(0.5, color="k", ls="--", lw=0.8)
    ax.axhline(1.0, color="k", ls=":", lw=0.8)
    ax.set_xlabel("a = sigma^2 / (2 kappa lambda)")
    ax.set_ylabel("fitted strong L2 order")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "jax_fig3_fitted_orders.pdf")
    plt.close(fig)
    return rows


def archive_outputs() -> None:
    import shutil

    archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
    print(f"Archived outputs to {archive}")


def main() -> None:
    print(f"Standalone GPU/JAX CIR benchmark suite, RUN_MODE={RUN_MODE}")
    print("JAX devices:", jax.devices())
    print(f"Output directory: {OUT_DIR}")
    streaming_parity_check()
    run_strong_error_suite()
    run_fig3_order_sweep()
    archive_outputs()
    print("Done.")


main()
